In [1]:
import sys, os
sys.path.append("..")
import pandas as pd
from src.loader import load_text

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
print(train.columns.tolist())
print(train.to_string())

['File Name', 'Aggrement Value', 'Aggrement Start Date', 'Aggrement End Date', 'Renewal Notice (Days)', 'Party One', 'Party Two']
                                                    File Name  Aggrement Value Aggrement Start Date Aggrement End Date  Renewal Notice (Days)                                               Party One                              Party Two
0  6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1             6500           20.05.2007         20.05.2008                   15.0  Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles                  GERALDINE Q. GALINATO
1         6683129-House-Rental-Contract-Geraldine-Galinato-v2             6500           20.05.2007         20.05.2008                   15.0  Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles                  GERALDINE Q. GALINATO
2                                 18325926-Rental-Agreement-1             4000           05.12.2008         31.11.2009                   90.0                  

In [2]:
for split in ["train", "test"]:
    print(split)
    for f in sorted(os.listdir(f"../data/{split}")):
        print("  ", repr(f))

train
   '18325926-Rental-Agreement-1.docx'
   '36199312-Rental-Agreement.png'
   '44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement.docx'
   '46239065-Standard-Rental-Agreement-Rental-With-Performance-Fee.docx'
   '47854715-RENTAL-AGREEMENT.docx'
   '50070534-RENTAL-AGREEMENT.docx'
   '54770958-Rental-Agreement.png'
   '54945838-Rental-Agreement.png'
   '6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1.docx'
   '6683129-House-Rental-Contract-Geraldine-Galinato-v2.docx'
test
   '156155545-Rental-Agreement-Kns-Home.pdf.docx'
   '228094620-Rental-Agreement.pdf.docx'
   '24158401-Rental-Agreement.png'
   '95980236-Rental-Agreement.png'


In [3]:
files = sorted(os.listdir("../data/train"))
docx_file = next(f for f in files if f.lower().endswith(".docx"))
png_file = next(f for f in files if f.lower().endswith(".png"))

print(load_text(f"../data/train/{docx_file}")[:1500])
print("=" * 60)
print(load_text(f"../data/train/{png_file}")[:1500])

RENTAL AGREEMENT
This deed of rental agreement executed at Bangalore this fifth day of December 2008 between MR.K.Kuttan S/o Kelu Aehari (Late) residing at site No 152 Geethalayam OMH colong S.M. Road 1st main, T.Dasarahalli, Bangalore-57. Here after called the lessor which term shall where the contest so admits mean and includes his legal heirs representatives executors and assign and Sri. P.M. Narayana Namboodri “Laxmi Leela” ground floor 3rd cross Ayyappa Nagar behind Ayyappa Temple, Jalahalli West, Bangalore - 15 referred as tenant which term shall wherever the context so admits mean and includes his legal heirs representations executors assign witnesses.
Whereas the lessor is the absolute owner after premises at site No. 8/20 S.M. Road, Jalahalli West, Bangalore - 15 and he has constructed approximate 500 sft area of Ilnd floor with A.C. Sheet roof with power light in the above said premises.
Where as the owner has offered to let out the premises for non residential purpose, where

In [2]:
import sys, os
sys.path.append("..")
import pandas as pd

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(train[["File Name", "Aggrement Value", "Party One", "Party Two"]].to_string())
print(test.columns.tolist())
print(test.to_string())

csv_names = set(train["File Name"])
folder_names = {os.path.splitext(f)[0] for f in os.listdir("../data/train")}
print("in CSV, not in train folder:", csv_names - folder_names)
print("in train folder, not in CSV:", folder_names - csv_names)

                                                    File Name  Aggrement Value                                               Party One                              Party Two
0  6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1             6500  Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles                  GERALDINE Q. GALINATO
1         6683129-House-Rental-Contract-Geraldine-Galinato-v2             6500  Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles                  GERALDINE Q. GALINATO
2                                 18325926-Rental-Agreement-1             4000                                            MR.K.Kuttan                P.M. Narayana Namboodri 
3                                   24158401-Rental-Agreement            12000                                              Hanumaiah                        Vishal Bhardwaj 
4                                   36199312-Rental-Agreement             3800                                               Balaj

In [1]:
import sys, os
sys.path.append("..")
import pandas as pd
from src.loader import load_text
from src.extractor import extract_fields, CSV_COLUMNS

train = pd.read_csv("../data/train.csv")

def find_file(folder, file_id):
    # match on the numeric ID, since CSV names have no extension
    for f in os.listdir(folder):
        if f.split("-")[0] == file_id:
            return os.path.join(folder, f)

file_id = "36199312"
row = train[train["File Name"].str.startswith(file_id)].iloc[0]
text = load_text(find_file("../data/train", file_id))

# check what the 'Aggrement Value' number actually is
print([l for l in text.splitlines() if "3800" in l or "3,800" in l])

pred = extract_fields(text)
for key, col in CSV_COLUMNS.items():
    print(f"{col:24} true={row[col]!r:35} pred={pred[key]!r}")

['The Rent is payable by the Lessees to the Lessor is a sum of Rs.3800/- (Rupees Thirty']
Aggrement Value          true=np.int64(3800)                      pred=3800
Aggrement Start Date     true='01.05.2010'                        pred='01.05.2010'
Aggrement End Date       true='31.04.2011'                        pred='31.03.2011'
Renewal Notice (Days)    true=np.float64(30.0)                    pred=30
Party One                true='Balaji.R '                         pred='Balaji.R'
Party Two                true='Kartheek R'                        pred='Kartheek R'


In [1]:
import sys, os
sys.path.append("..")
import pandas as pd

from src.evaluate import run, norm
pred_df, true_df, recall = run("../data/train.csv", "../data/train")
print(pd.Series(recall).round(2))

# see every miss
for col in recall:
    for t, p, name in zip(true_df[col], pred_df[col], pred_df["File Name"]):
        if norm(t) != norm(p):
            print(f"{name[:30]:30} | {col:22} | true={t!r} pred={p!r}")

skipping (file not in folder): 24158401-Rental-Agreement
Aggrement Value          0.67
Aggrement Start Date     0.67
Aggrement End Date       0.33
Renewal Notice (Days)    0.44
Party One                0.33
Party Two                0.56
dtype: float64
44737744-Maddireddy-Bhargava-R | Aggrement Value        | true=3000 pred=9000
54770958-Rental-Agreement      | Aggrement Value        | true=8000 pred=5500
54945838-Rental-Agreement      | Aggrement Value        | true=5500 pred=8000
44737744-Maddireddy-Bhargava-R | Aggrement Start Date   | true='20.09.2010' pred=nan
54770958-Rental-Agreement      | Aggrement Start Date   | true='01.04.2011' pred='20.04.2011'
54945838-Rental-Agreement      | Aggrement Start Date   | true='21.04.2011' pred='01.04.2011'
18325926-Rental-Agreement-1    | Aggrement End Date     | true='31.11.2009' pred='04.11.2009'
36199312-Rental-Agreement      | Aggrement End Date     | true='31.04.2011' pred='31.03.2011'
44737744-Maddireddy-Bhargava-R | Aggrement End Date  

In [2]:
import os
from src.loader import load_text
from src.evaluate import find_file

pred_df.to_csv("../outputs/train_predictions_v1.csv", index=False)

# 1. the Party One misses that were cut off
col = "Party One"
for t, p, name in zip(true_df[col], pred_df[col], pred_df["File Name"]):
    if norm(t) != norm(p):
        print(name[:30], "| true:", repr(t), "| pred:", repr(p))

# 2. are the two docs swapped? look at who the parties are
for fid in ["54770958", "54945838"]:
    t = load_text(find_file("../data/train", fid))
    print("\n", fid, "\n", t[:900], "\n", "=" * 60)

# 3. lines about notice/rent/dates in the problem docs
def show(fid, words):
    t = load_text(find_file("../data/train", fid))
    print("\n###", fid, "| chars:", len(t))
    for line in t.splitlines():
        if any(w in line.lower() for w in words):
            print("  ", line[:200])

show("44737744", ["rent", "month", "2010", "2011"])
show("6683127", ["notice", "days"])
show("47854715", ["notice", "days", "month"])

6683127-House-Rental-Contract- | true: 'Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles' | pred: 'Antonio Levy S. Ingles. Jr. and/or Mary Rose C. Ingles'
6683129-House-Rental-Contract- | true: 'Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles' | pred: 'Antonio Levy S. Ingles. Jr. and/or Mary Rose C. Ingles'
18325926-Rental-Agreement-1 | true: 'MR.K.Kuttan ' | pred: 'K.Kuttan'
44737744-Maddireddy-Bhargava-R | true: 'M.V.V. VIJAYA SHANKAR' | pred: 'M. V, V« VI JAY A SHANKAR'
54770958-Rental-Agreement | true: 'K. Parthasarathy' | pred: 'Asha Ramesh, Ramesh.K.N'
54945838-Rental-Agreement | true: 'Asha Ramesh & Ramesh K.N' | pred: 'Prof. K. Parthasarathy'

 54770958 
 RENTAL AGREEMENT

This RENTAL AGREEMENT is made & executed on 20 April 2011 between: Mrs.Asha Ramesh &Mr.Ramesh.K.N. Residing at 1st
Floor, 82A, Vysya Bank Colony, ShantiniketanLayout, Arakere Post, Bannerghatta Road, Bangalore-560076, henceforth calld
the LESSOR (OWNERS), which term include their heirs, administr

In [3]:
import pandas as pd
from src.evaluate import norm
from src.extractor import CSV_COLUMNS

a, b = "54770958-Rental-Agreement", "54945838-Rental-Agreement"

adj_true = true_df.copy()
ia = adj_true.index[adj_true["File Name"] == a][0]
ib = adj_true.index[adj_true["File Name"] == b][0]
cols = list(CSV_COLUMNS.values())
adj_true.loc[ia, cols], adj_true.loc[ib, cols] = (
    true_df.loc[ib, cols].values, true_df.loc[ia, cols].values)

adj_recall = {}
for col in cols:
    hits = [norm(t) == norm(p) for t, p in zip(adj_true[col], pred_df[col])]
    adj_recall[col] = sum(hits) / len(hits)

print(pd.DataFrame({"strict": recall, "adjusted": adj_recall}).round(2))

                       strict  adjusted
Aggrement Value          0.67      0.89
Aggrement Start Date     0.67      0.78
Aggrement End Date       0.33      0.44
Renewal Notice (Days)    0.44      0.56
Party One                0.33      0.33
Party Two                0.56      0.56


In [1]:
import sys, os
sys.path.append("..")
import pandas as pd
from src.evaluate import run, norm

pred_df, true_df, recall = run("../data/train.csv", "../data/train")
pred_df.to_csv("../outputs/train_predictions_v2.csv", index=False)
print(pd.Series(recall).round(2))

skipping (file not in folder): 24158401-Rental-Agreement
Aggrement Value          0.67
Aggrement Start Date     0.67
Aggrement End Date       0.33
Renewal Notice (Days)    0.67
Party One                0.33
Party Two                0.56
dtype: float64


In [1]:
import sys, os
sys.path.append("..")
import pandas as pd
from src.evaluate import run, norm
from src.extractor import CSV_COLUMNS

# 1. Evaluation (isme kuch minute lagenge, ek hi baar chalao)
pred_df, true_df, recall = run("../data/train.csv", "../data/train")
pred_df.to_csv("../outputs/train_predictions_v2.csv", index=False)

# 2. Adjusted recall (54770958 aur 54945838 ke labels copy mein swap)
a, b = "54770958-Rental-Agreement", "54945838-Rental-Agreement"
adj_true = true_df.copy()
ia = adj_true.index[adj_true["File Name"] == a][0]
ib = adj_true.index[adj_true["File Name"] == b][0]
cols = list(CSV_COLUMNS.values())
adj_true.loc[ia, cols], adj_true.loc[ib, cols] = (
    true_df.loc[ib, cols].values, true_df.loc[ia, cols].values)

adj_recall = {}
for col in cols:
    hits = [norm(t) == norm(p) for t, p in zip(adj_true[col], pred_df[col])]
    adj_recall[col] = sum(hits) / len(hits)

print(pd.DataFrame({"strict": recall, "adjusted": adj_recall}).round(2))

skipping (file not in folder): 24158401-Rental-Agreement
                       strict  adjusted
Aggrement Value          0.67      0.89
Aggrement Start Date     0.67      0.78
Aggrement End Date       0.33      0.44
Renewal Notice (Days)    0.67      0.89
Party One                0.33      0.44
Party Two                0.56      0.67


In [2]:
import sys, json
sys.path.append("..")
import pandas as pd
from src.evaluate import run, norm

test_pred, test_true, test_recall = run("../data/test.csv", "../data/test")
test_pred.to_csv("../outputs/test_predictions.csv", index=False)
print(pd.Series(test_recall).round(2))

# misses, for the README
for col in test_recall:
    for t, p, name in zip(test_true[col], test_pred[col], test_pred["File Name"]):
        if norm(t) != norm(p):
            print(f"{name[:25]:25} | {col:22} | true={t!r} pred={p!r}")

# save all scores (recall and adj_recall exist only if you didn't restart since the last cell)
json.dump({"train_strict": recall, "train_adjusted": adj_recall, "test": test_recall},
          open("../outputs/recall_scores.json", "w"), indent=2)

Aggrement Value          1.00
Aggrement Start Date     1.00
Aggrement End Date       0.75
Renewal Notice (Days)    1.00
Party One                0.75
Party Two                0.25
dtype: float64
95980236-Rental-Agreement | Aggrement End Date     | true='31.03.2011' pred='28.02.2011'
24158401-Rental-Agreement | Party One              | true='Hanumaiah ' pred='Sri Hanumaiah'
24158401-Rental-Agreement | Party Two              | true=' Vishal Bhardwaj ' pred='Sri Vishal Bhardwaj'
156155545-Rental-Agreemen | Party Two              | true=' VYSHNAVI DAIRY SPECIALITIES Private Ltd' pred='SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.'
228094620-Rental-Agreemen | Party Two              | true='.B.Kishore ' pred='B.Kishore'


In [2]:
import sys, json
sys.path.append("..")
import pandas as pd
from src.evaluate import run, norm

pred_df, true_df, recall = run("../data/train.csv", "../data/train")
pred_df.to_csv("../outputs/train_predictions_v3.csv", index=False)
json.dump(recall, open("../outputs/train_recall_v3.json", "w"), indent=2)
print(pd.Series(recall).round(2))

skipping (file not in folder): 24158401-Rental-Agreement
Aggrement Value          0.67
Aggrement Start Date     0.67
Aggrement End Date       0.33
Renewal Notice (Days)    0.67
Party One                0.33
Party Two                0.56
dtype: float64


In [3]:
test_pred, test_true, test_recall = run("../data/test.csv", "../data/test")
test_pred.to_csv("../outputs/test_predictions_v3.csv", index=False)
json.dump(test_recall, open("../outputs/test_recall_v3.json", "w"), indent=2)
print(pd.Series(test_recall).round(2))

for col in test_recall:
    for t, p, name in zip(test_true[col], test_pred[col], test_pred["File Name"]):
        if norm(t) != norm(p):
            print(f"{name[:25]:25} | {col:22} | true={t!r} pred={p!r}")

Aggrement Value          1.00
Aggrement Start Date     1.00
Aggrement End Date       0.75
Renewal Notice (Days)    1.00
Party One                1.00
Party Two                0.50
dtype: float64
95980236-Rental-Agreement | Aggrement End Date     | true='31.03.2011' pred='28.02.2011'
156155545-Rental-Agreemen | Party Two              | true=' VYSHNAVI DAIRY SPECIALITIES Private Ltd' pred='SRI VYSHNAVI DAIRY SPECIALITIES Private Ltd.'
228094620-Rental-Agreemen | Party Two              | true='.B.Kishore ' pred='B.Kishore'
